# Exploration: why this marketplace cannot afford a slow payback

This notebook is the reasoning behind the headline result. It runs on the real Olist
order data; the acquisition channel and media spend used elsewhere in the project are
simulated and are not touched here.

The question it answers: **how long does a customer stay, and what does that leave the
business able to spend to acquire one?**

In [1]:
import sys
sys.path.insert(0, "src")

import duckdb
import pandas as pd

from unit_economics.ingest import DATA_DIR, download, verify

download()
verify()
print({k: f"{v:,}" for k, v in verify().items()})

{'olist_customers_dataset': '99,441', 'olist_orders_dataset': '99,441', 'olist_order_items_dataset': '112,650', 'olist_order_payments_dataset': '103,886'}


## 1. The identity trap

Olist ships two customer keys. `customer_id` is issued **per order**; `customer_unique_id`
identifies the **person**. Grouping by the wrong one makes repeat purchase impossible to
observe, because every order becomes a new customer.

In [2]:
con = duckdb.connect()
customers = con.execute(
    f"SELECT * FROM read_csv_auto('{DATA_DIR}/olist_customers_dataset.csv', header=true)"
).df()

print(f"customer_id        {customers['customer_id'].nunique():,}")
print(f"customer_unique_id {customers['customer_unique_id'].nunique():,}")
print(f"\nthe gap, {customers['customer_id'].nunique() - customers['customer_unique_id'].nunique():,} "
      "rows, is exactly the repeat-purchase behaviour a naive grouping destroys")

customer_id        99,441
customer_unique_id 96,096

the gap, 3,345 rows, is exactly the repeat-purchase behaviour a naive grouping destroys


## 2. How often does a customer come back?

In [3]:
orders = con.execute(
    f"""
    SELECT c.customer_unique_id, COUNT(DISTINCT o.order_id) AS orders
    FROM read_csv_auto('{DATA_DIR}/olist_orders_dataset.csv', header=true) o
    JOIN read_csv_auto('{DATA_DIR}/olist_customers_dataset.csv', header=true) c
      ON o.customer_id = c.customer_id
    WHERE o.order_status = 'delivered'
    GROUP BY c.customer_unique_id
    """
).df()

distribution = (
    orders.groupby("orders").size().rename("customers").reset_index()
)
distribution["pct"] = 100 * distribution["customers"] / distribution["customers"].sum()
print(distribution.head(6).to_string(index=False))
print(f"\nsingle-purchase customers: {distribution.iloc[0]['pct']:.2f}%")

 orders  customers       pct
      1      90557 96.999722
      2       2573  2.756057
      3        181  0.193877
      4         28  0.029992
      5          9  0.009640
      6          5  0.005356

single-purchase customers: 97.00%


**97% buy exactly once.**

That single number decides the shape of the whole analysis. There is no second order to
recover acquisition cost from, so CAC has to clear on the first purchase, and the first
order's contribution margin becomes a hard ceiling rather than a soft target.

## 3. What does one order actually leave the marketplace?

Olist is a marketplace: it keeps a commission, not the GMV. Modelling margin on GMV would
overstate the economics by roughly the inverse of the take rate.

In [4]:
from unit_economics.config import MARGIN
from unit_economics.metrics import break_even_cac, contribution_margin

gmv = con.execute(
    f"""
    SELECT SUM(i.price) / COUNT(DISTINCT c.customer_unique_id) AS gmv_per_customer
    FROM read_csv_auto('{DATA_DIR}/olist_order_items_dataset.csv', header=true) i
    JOIN read_csv_auto('{DATA_DIR}/olist_orders_dataset.csv', header=true) o USING (order_id)
    JOIN read_csv_auto('{DATA_DIR}/olist_customers_dataset.csv', header=true) c
      ON o.customer_id = c.customer_id
    WHERE o.order_status = 'delivered'
    """
).df()["gmv_per_customer"].iloc[0]

margin = contribution_margin(gmv, MARGIN.take_rate, MARGIN.psp_fee_rate, MARGIN.variable_support_cost)

retail_margin = 0.30  # what a retailer with this basket might keep
retail_view = gmv * retail_margin

print(f"GMV per customer                 R$ {gmv:8.2f}")
print(f"take rate (assumption)           {MARGIN.take_rate:11.0%}")
print(f"marketplace contribution margin  R$ {margin:8.2f}")
print()
print("Modelled as a RETAILER instead, keeping a 30% margin on the basket:")
print(f"  margin per customer            R$ {retail_view:8.2f}")
print(f"  overstatement                  {retail_view / margin:8.1f}x")
print()
print("The marketplace never holds the basket. It holds the commission, and pays")
print("the payment processor on the full transaction. That is the whole difference.")

GMV per customer                 R$   141.62
take rate (assumption)                   15%
marketplace contribution margin  R$    16.20

Modelled as a RETAILER instead, keeping a 30% margin on the basket:
  margin per customer            R$    42.49
  overstatement                       2.6x

The marketplace never holds the basket. It holds the commission, and pays
the payment processor on the full transaction. That is the whole difference.


## 4. The spending ceiling, at several take rates

In [5]:
rows = []
for take_rate in (0.20, 0.15, 0.125, 0.10, 0.08):
    m = contribution_margin(gmv, take_rate, MARGIN.psp_fee_rate, MARGIN.variable_support_cost)
    rows.append({
        "take_rate": f"{take_rate:.1%}",
        "margin_per_customer": round(m, 2),
        "break_even_cac": round(break_even_cac(m), 2),
        "cac_ceiling_at_3x": round(break_even_cac(m, 3.0), 2),
    })
print(pd.DataFrame(rows).to_string(index=False))

take_rate  margin_per_customer  break_even_cac  cac_ceiling_at_3x
    20.0%                23.28           23.28               7.76
    15.0%                16.20           16.20               5.40
    12.5%                12.66           12.66               4.22
    10.0%                 9.12            9.12               3.04
     8.0%                 6.29            6.29               2.10


## What this means for the decision

At the assumed 15% take rate the business can pay about **R$ 16 to acquire a customer** and
break even, or about **R$ 5.40** to hit the 3x that media teams usually plan against.

The take rate is the most sensitive input and nobody outside Olist knows it. Below roughly
12%, paid search and paid social stop paying for themselves entirely — so pinning that
number down is the first thing to do before acting on any channel recommendation.

Full channel analysis: `python -m unit_economics.pipeline`.